In [1]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# Load processed datasets

DATA_DIR = Path("../data/processed")

X_train = pd.read_pickle(DATA_DIR / "X_train.pkl")
X_val = pd.read_pickle(DATA_DIR / "X_val.pkl")
X_test = pd.read_pickle(DATA_DIR / "X_test.pkl")

y_train = pd.read_pickle(DATA_DIR / "y_train.pkl")
y_val = pd.read_pickle(DATA_DIR / "y_val.pkl")
y_test = pd.read_pickle(DATA_DIR / "y_test.pkl")

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (413378, 421)
X_val: (88581, 421)
X_test: (88581, 421)

y_train: (413378,)
y_val: (88581,)
y_test: (88581,)


In [3]:
# Check class distribution

class_counts = y_train.value_counts().sort_index()

print("Class distribution:")
print(class_counts)

print("\nPercentage distribution:")

print((y_train.value_counts(normalize=True) * 100).sort_index())

Class distribution:
isFraud
0    398840
1     14538
Name: count, dtype: int64

Percentage distribution:
isFraud
0    96.483122
1     3.516878
Name: proportion, dtype: float64


In [4]:
# Calculate class weights for imbalanced fraud data

from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
)

print("Class weights:")
print("Legitimate (0):", class_weights[0].item())
print("Fraud (1):", class_weights[1].item())

Class weights:
Legitimate (0): 0.518225371837616
Fraud (1): 14.217155456542969


In [5]:
# Define the DNN model

class FraudDNN(nn.Module):
    def __init__(self, input_size):
        super(FraudDNN, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)


# Number of input features
input_size = X_train.shape[1]

# Create model
model = FraudDNN(input_size)

print(model)
print("\nInput features:", input_size)

FraudDNN(
  (network): Sequential(
    (0): Linear(in_features=421, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.2, inplace=False)
    (9): Linear(in_features=64, out_features=1, bias=True)
  )
)

Input features: 421


In [6]:
# Check for non-numeric columns

non_numeric = X_train.select_dtypes(exclude=[np.number]).columns

print("Number of non-numeric columns:", len(non_numeric))

print("\nNon-numeric columns:")
print(list(non_numeric))

Number of non-numeric columns: 29

Non-numeric columns:
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [7]:
# Convert categorical columns to numerical values
# Use the combined train + validation + test categories
# so that all datasets use the same encoding.

categorical_columns = X_train.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("Categorical columns:", len(categorical_columns))

# Combine the datasets temporarily for consistent encoding
X_all = pd.concat(
    [X_train, X_val, X_test],
    axis=0
)

# Convert categorical columns to category codes
for col in categorical_columns:
    X_all[col] = X_all[col].astype("category").cat.codes

# Split them back
n_train = len(X_train)
n_val = len(X_val)

X_train_dnn = X_all.iloc[:n_train].copy()
X_val_dnn = X_all.iloc[n_train:n_train + n_val].copy()
X_test_dnn = X_all.iloc[n_train + n_val:].copy()

print("X_train_dnn:", X_train_dnn.shape)
print("X_val_dnn:", X_val_dnn.shape)
print("X_test_dnn:", X_test_dnn.shape)

print("\nRemaining non-numeric columns:",
      len(X_train_dnn.select_dtypes(exclude=[np.number]).columns))

Categorical columns: 29
X_train_dnn: (413378, 421)
X_val_dnn: (88581, 421)
X_test_dnn: (88581, 421)

Remaining non-numeric columns: 0


In [8]:
# Convert numerical DNN data to PyTorch tensors

X_train_tensor = torch.tensor(
    X_train_dnn.values,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.float32
).view(-1, 1)

X_val_tensor = torch.tensor(
    X_val_dnn.values,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val.values,
    dtype=torch.float32
).view(-1, 1)

X_test_tensor = torch.tensor(
    X_test_dnn.values,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test.values,
    dtype=torch.float32
).view(-1, 1)

print("Training:", X_train_tensor.shape, y_train_tensor.shape)
print("Validation:", X_val_tensor.shape, y_val_tensor.shape)
print("Test:", X_test_tensor.shape, y_test_tensor.shape)

Training: torch.Size([413378, 421]) torch.Size([413378, 1])
Validation: torch.Size([88581, 421]) torch.Size([88581, 1])
Test: torch.Size([88581, 421]) torch.Size([88581, 1])


In [9]:
# Create PyTorch DataLoaders

BATCH_SIZE = 512

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Batch size:", BATCH_SIZE)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Batch size: 512
Training batches: 808
Validation batches: 174
Test batches: 174


In [10]:
# Training configuration

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

# Weighted loss for class imbalance
pos_weight = torch.tensor(
    [class_weights[1] / class_weights[0]],
    dtype=torch.float32
).to(device)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

# Adam optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Device:", device)
print("Positive class weight:", pos_weight.item())
print("Learning rate:", 0.001)

Device: cpu
Positive class weight: 27.434309005737305
Learning rate: 0.001


In [11]:
# Train the DNN

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()

    total_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(X_batch)

        # Calculate loss
        loss = criterion(outputs, y_batch)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # Validation
    model.eval()

    val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"- Train Loss: {avg_train_loss:.4f} "
        f"- Val Loss: {avg_val_loss:.4f}"
    )

Epoch 1/5 - Train Loss: nan - Val Loss: nan
Epoch 2/5 - Train Loss: nan - Val Loss: nan
Epoch 3/5 - Train Loss: nan - Val Loss: nan
Epoch 4/5 - Train Loss: nan - Val Loss: nan
Epoch 5/5 - Train Loss: nan - Val Loss: nan


In [12]:
# Check for NaN and infinite values

print("NaN values in X_train:", torch.isnan(X_train_tensor).sum().item())
print("NaN values in X_val:", torch.isnan(X_val_tensor).sum().item())

print("Infinite values in X_train:",
      torch.isinf(X_train_tensor).sum().item())

print("Infinite values in X_val:",
      torch.isinf(X_val_tensor).sum().item())

print("\nMaximum absolute training value:",
      torch.max(torch.abs(X_train_tensor)).item())

print("Minimum training value:",
      torch.min(X_train_tensor).item())

NaN values in X_train: 68357712
NaN values in X_val: 15365202
Infinite values in X_train: 0
Infinite values in X_val: 0

Maximum absolute training value: nan
Minimum training value: nan


In [13]:
# Find columns containing missing values

nan_counts = X_train_dnn.isna().sum()

nan_columns = nan_counts[nan_counts > 0].sort_values(ascending=False)

print("Number of columns containing NaN:", len(nan_columns))

print("\nTop columns with missing values:")
print(nan_columns.head(20))

Number of columns containing NaN: 331

Top columns with missing values:
D13      369305
D14      368410
D12      366089
id_04    364211
id_03    364211
D6       360808
id_10    357191
id_09    357191
D9       357191
D8       357191
id_32    351568
id_14    349620
V155     348208
V154     348208
V141     348208
V156     348208
V157     348208
V140     348208
V150     348208
V158     348208
dtype: int64


In [14]:
# Handle missing values using training-set medians

X_train_dnn = X_train_dnn.copy()
X_val_dnn = X_val_dnn.copy()
X_test_dnn = X_test_dnn.copy()

# Calculate medians only from training data
train_medians = X_train_dnn.median(numeric_only=True)

# Fill missing values using training medians
X_train_dnn = X_train_dnn.fillna(train_medians)
X_val_dnn = X_val_dnn.fillna(train_medians)
X_test_dnn = X_test_dnn.fillna(train_medians)

print("NaN values after imputation:")
print("Training:", X_train_dnn.isna().sum().sum())
print("Validation:", X_val_dnn.isna().sum().sum())
print("Test:", X_test_dnn.isna().sum().sum())

NaN values after imputation:
Training: 0
Validation: 0
Test: 0


In [15]:
# Recreate PyTorch tensors after missing-value imputation

X_train_tensor = torch.tensor(
    X_train_dnn.values,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.float32
).view(-1, 1)

X_val_tensor = torch.tensor(
    X_val_dnn.values,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val.values,
    dtype=torch.float32
).view(-1, 1)

X_test_tensor = torch.tensor(
    X_test_dnn.values,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test.values,
    dtype=torch.float32
).view(-1, 1)

print("Training:", X_train_tensor.shape, y_train_tensor.shape)
print("Validation:", X_val_tensor.shape, y_val_tensor.shape)
print("Test:", X_test_tensor.shape, y_test_tensor.shape)

Training: torch.Size([413378, 421]) torch.Size([413378, 1])
Validation: torch.Size([88581, 421]) torch.Size([88581, 1])
Test: torch.Size([88581, 421]) torch.Size([88581, 1])


In [16]:
# Final numerical check before training

print("NaN in training:", torch.isnan(X_train_tensor).sum().item())
print("NaN in validation:", torch.isnan(X_val_tensor).sum().item())
print("NaN in test:", torch.isnan(X_test_tensor).sum().item())

print("Inf in training:", torch.isinf(X_train_tensor).sum().item())
print("Inf in validation:", torch.isinf(X_val_tensor).sum().item())
print("Inf in test:", torch.isinf(X_test_tensor).sum().item())

print("\nMaximum value:", torch.max(X_train_tensor).item())
print("Minimum value:", torch.min(X_train_tensor).item())

NaN in training: 0
NaN in validation: 0
NaN in test: 0
Inf in training: 0
Inf in validation: 0
Inf in test: 0

Maximum value: 10437996.0
Minimum value: -660.0


In [17]:
from sklearn.preprocessing import StandardScaler

# Create scaler
scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train_dnn)

# Apply the same scaler to validation and test
X_val_scaled = scaler.transform(X_val_dnn)
X_test_scaled = scaler.transform(X_test_dnn)

print("Training mean:", np.mean(X_train_scaled))
print("Training std:", np.std(X_train_scaled))

print("NaN in scaled training:", np.isnan(X_train_scaled).sum())
print("Inf in scaled training:", np.isinf(X_train_scaled).sum())

Training mean: -1.5566e-08
Training std: 0.9988116
NaN in scaled training: 0
Inf in scaled training: 0


In [18]:
# Convert scaled data to PyTorch tensors

X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val_scaled,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

print("Training:", X_train_tensor.shape)
print("Validation:", X_val_tensor.shape)
print("Test:", X_test_tensor.shape)

Training: torch.Size([413378, 421])
Validation: torch.Size([88581, 421])
Test: torch.Size([88581, 421])


In [19]:
# Recreate DataLoaders using scaled tensors

BATCH_SIZE = 512

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Batch size:", BATCH_SIZE)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Batch size: 512
Training batches: 808
Validation batches: 174
Test batches: 174


In [20]:
# Recreate the DNN model

model = FraudDNN(input_size=421).to(device)

# Weighted loss for fraud class imbalance
pos_weight = torch.tensor(
    [class_weights[1] / class_weights[0]],
    dtype=torch.float32
).to(device)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("New DNN created")
print("Device:", device)
print("Positive class weight:", pos_weight.item())

New DNN created
Device: cpu
Positive class weight: 27.434309005737305


TESTING

In [21]:
# Test one batch before full training

model.train()

X_batch, y_batch = next(iter(train_loader))

X_batch = X_batch.to(device)
y_batch = y_batch.to(device)

outputs = model(X_batch)

loss = criterion(outputs, y_batch)

print("Input shape:", X_batch.shape)
print("Output shape:", outputs.shape)
print("Loss:", loss.item())

print("Output contains NaN:", torch.isnan(outputs).any().item())
print("Loss is NaN:", torch.isnan(loss).item())

Input shape: torch.Size([512, 421])
Output shape: torch.Size([512, 1])
Loss: 1.5863484144210815
Output contains NaN: False
Loss is NaN: False


In [22]:
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"- Train Loss: {avg_train_loss:.4f} "
        f"- Val Loss: {avg_val_loss:.4f}"
    )

Epoch 1/5 - Train Loss: 0.8964 - Val Loss: 0.8858
Epoch 2/5 - Train Loss: 0.8218 - Val Loss: 0.8819
Epoch 3/5 - Train Loss: 0.7863 - Val Loss: 0.8788
Epoch 4/5 - Train Loss: 0.7592 - Val Loss: 0.8930
Epoch 5/5 - Train Loss: 0.7409 - Val Loss: 0.9322


In [23]:
# Evaluate DNN on validation data

model.eval()

val_probabilities = []
val_actual = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        probabilities = torch.sigmoid(outputs)

        val_probabilities.extend(
            probabilities.cpu().numpy().flatten()
        )

        val_actual.extend(
            y_batch.numpy().flatten()
        )

val_probabilities = np.array(val_probabilities)
val_actual = np.array(val_actual)

# Basic metrics
val_roc_auc = roc_auc_score(
    val_actual,
    val_probabilities
)

val_pr_auc = average_precision_score(
    val_actual,
    val_probabilities
)

print("Validation ROC-AUC:", round(val_roc_auc, 4))
print("Validation PR-AUC:", round(val_pr_auc, 4))

Validation ROC-AUC: 0.8494
Validation PR-AUC: 0.4429


In [24]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.05, 0.96, 0.05)

results = []

for threshold in thresholds:
    predictions = (val_probabilities >= threshold).astype(int)

    precision = precision_score(
        val_actual,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        val_actual,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        val_actual,
        predictions,
        zero_division=0
    )

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results = pd.DataFrame(results)

print(
    threshold_results.sort_values(
        "f1",
        ascending=False
    ).to_string(index=False)
)

 threshold  precision   recall       f1
      0.90   0.527477 0.384944 0.445078
      0.95   0.665101 0.325773 0.437335
      0.85   0.441098 0.433268 0.437148
      0.80   0.370749 0.476660 0.417086
      0.75   0.314130 0.510848 0.389035
      0.70   0.270630 0.555227 0.363891
      0.65   0.232558 0.595003 0.334411
      0.60   0.199959 0.636423 0.304307
      0.55   0.172885 0.673241 0.275121
      0.50   0.148069 0.701841 0.244545
      0.45   0.129183 0.728468 0.219449
      0.40   0.113300 0.754438 0.197013
      0.35   0.098241 0.789612 0.174742
      0.30   0.081008 0.825444 0.147537
      0.25   0.068289 0.864234 0.126577
      0.20   0.058616 0.895792 0.110032
      0.15   0.050533 0.928994 0.095852
      0.10   0.043219 0.962525 0.082724
      0.05   0.037196 0.985207 0.071685


In [25]:
best_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = float(best_row["threshold"])

print("Selected threshold:", best_threshold)
print("Validation Precision:", best_row["precision"])
print("Validation Recall:", best_row["recall"])
print("Validation F1:", best_row["f1"])

Selected threshold: 0.9000000000000001
Validation Precision: 0.5274774774774775
Validation Recall: 0.38494411571334647
Validation F1: 0.4450779171417712


THRESHOLD 


In [26]:
# Evaluate DNN on the untouched test set

model.eval()

test_probabilities = []
test_actual = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        probabilities = torch.sigmoid(outputs)

        test_probabilities.extend(
            probabilities.cpu().numpy().flatten()
        )

        test_actual.extend(
            y_batch.numpy().flatten()
        )

test_probabilities = np.array(test_probabilities)
test_actual = np.array(test_actual)

# Apply the fixed validation threshold
test_predictions = (
    test_probabilities >= best_threshold
).astype(int)

# Metrics
test_roc_auc = roc_auc_score(
    test_actual,
    test_probabilities
)

test_pr_auc = average_precision_score(
    test_actual,
    test_probabilities
)

print("Test ROC-AUC:", round(test_roc_auc, 4))
print("Test PR-AUC:", round(test_pr_auc, 4))

print("\nClassification Report:")
print(
    classification_report(
        test_actual,
        test_predictions,
        target_names=["Legitimate", "Fraud"],
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        test_actual,
        test_predictions
    )
)

Test ROC-AUC: 0.8291
Test PR-AUC: 0.2332

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.98      0.97      0.97     85498
       Fraud       0.33      0.42      0.37      3083

    accuracy                           0.95     88581
   macro avg       0.66      0.69      0.67     88581
weighted avg       0.96      0.95      0.95     88581

Confusion Matrix:
[[82911  2587]
 [ 1792  1291]]


In [33]:
import numpy as np
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Make sure data types are correct
# --------------------------------------------------

if isinstance(X_train_dnn, np.ndarray):
    X_train_dnn = pd.DataFrame(X_train_dnn)

if isinstance(y_train, np.ndarray):
    y_train = pd.Series(y_train)

print("X_train_dnn type:", type(X_train_dnn))
print("y_train type:", type(y_train))
print("X_train_dnn shape:", X_train_dnn.shape)
print("y_train shape:", y_train.shape)


# --------------------------------------------------
# 2. Combine X and y
# --------------------------------------------------

train_df = X_train_dnn.copy()
train_df["isFraud"] = y_train.to_numpy()


# --------------------------------------------------
# 3. Separate legitimate and fraud transactions
# --------------------------------------------------

legit_df = train_df[
    train_df["isFraud"] == 0
].copy()

fraud_df = train_df[
    train_df["isFraud"] == 1
].copy()

# Shuffle
legit_df = legit_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

fraud_df = fraud_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


# --------------------------------------------------
# 4. Manually split legitimate transactions
#    into 5 pandas DataFrames
# --------------------------------------------------

num_clients = 5

legit_splits = []

total_legit = len(legit_df)
split_size = total_legit // num_clients

for i in range(num_clients):

    start_idx = i * split_size

    if i == num_clients - 1:
        end_idx = total_legit
    else:
        end_idx = (i + 1) * split_size

    client_legit = legit_df.iloc[
        start_idx:end_idx
    ].copy()

    legit_splits.append(client_legit)


# Check the split types
print("\nLegitimate client data types:")

for i, df in enumerate(legit_splits):
    print(
        f"Client {i}: {type(df)}, shape={df.shape}"
    )


# --------------------------------------------------
# 5. Allocate fraud transactions differently
#    across the simulated banks
# --------------------------------------------------

fraud_distribution = [
    0.05,
    0.10,
    0.15,
    0.25,
    0.45
]

fraud_counts = [
    int(len(fraud_df) * ratio)
    for ratio in fraud_distribution
]

# Make sure every fraud transaction is assigned
fraud_counts[-1] = (
    len(fraud_df) - sum(fraud_counts[:-1])
)


# --------------------------------------------------
# 6. Create federated data directory
# --------------------------------------------------

base_dir = Path("../federated/data")

base_dir.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------
# 7. Create the 5 simulated bank datasets
# --------------------------------------------------

start = 0

for client_id in range(num_clients):

    # ------------------------------
    # Fraud transactions
    # ------------------------------

    fraud_count = fraud_counts[client_id]

    client_fraud = fraud_df.iloc[
        start:start + fraud_count
    ].copy()

    start += fraud_count


    # ------------------------------
    # Legitimate transactions
    # ------------------------------

    client_legit = legit_splits[client_id].copy()


    # ------------------------------
    # Combine
    # ------------------------------

    client_df = pd.concat(
        [
            client_legit,
            client_fraud
        ],
        axis=0,
        ignore_index=True
    )


    # Shuffle client data
    client_df = client_df.sample(
        frac=1,
        random_state=42 + client_id
    ).reset_index(drop=True)


    # ------------------------------
    # Separate X and y
    # ------------------------------

    X_client = client_df.drop(
        columns=["isFraud"]
    )

    y_client = client_df["isFraud"]


    # ------------------------------
    # Create client folder
    # ------------------------------

    client_dir = (
        base_dir / f"client_{client_id}"
    )

    client_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    # ------------------------------
    # Save
    # ------------------------------

    X_client.to_pickle(
        client_dir / "X.pkl"
    )

    y_client.to_pickle(
        client_dir / "y.pkl"
    )


    # ------------------------------
    # Display statistics
    # ------------------------------

    fraud_actual = int(y_client.sum())

    legitimate_actual = int(
        (y_client == 0).sum()
    )

    fraud_rate = (
        y_client.mean() * 100
    )

    print(
        f"\nClient {client_id}"
    )

    print(
        f"  Total: {len(client_df)}"
    )

    print(
        f"  Legitimate: {legitimate_actual}"
    )

    print(
        f"  Fraud: {fraud_actual}"
    )

    print(
        f"  Fraud rate: {fraud_rate:.2f}%"
    )

X_train_dnn type: <class 'pandas.DataFrame'>
y_train type: <class 'pandas.Series'>
X_train_dnn shape: (413378, 421)
y_train shape: (413378,)

Legitimate client data types:
Client 0: <class 'pandas.DataFrame'>, shape=(79768, 422)
Client 1: <class 'pandas.DataFrame'>, shape=(79768, 422)
Client 2: <class 'pandas.DataFrame'>, shape=(79768, 422)
Client 3: <class 'pandas.DataFrame'>, shape=(79768, 422)
Client 4: <class 'pandas.DataFrame'>, shape=(79768, 422)

Client 0
  Total: 80494
  Legitimate: 79768
  Fraud: 726
  Fraud rate: 0.90%

Client 1
  Total: 81221
  Legitimate: 79768
  Fraud: 1453
  Fraud rate: 1.79%

Client 2
  Total: 81948
  Legitimate: 79768
  Fraud: 2180
  Fraud rate: 2.66%

Client 3
  Total: 83402
  Legitimate: 79768
  Fraud: 3634
  Fraud rate: 4.36%

Client 4
  Total: 86313
  Legitimate: 79768
  Fraud: 6545
  Fraud rate: 7.58%
